# ETTh1 / ETTh2 эксперименты: LSTM + Optuna, Informer + Optuna, LLM-агент

В этом ноутбуке запускаются три подхода к задаче прогноза временных рядов на датасетах **ETTh1** и **ETTh2**:

1. **LSTM + Optuna**  
   Поиск гиперпараметров LSTM с помощью Optuna и обучение лучшей модели.  
   Сохраняются:
   - метрики качества (MSE);
   - ресурсные метрики (время, память, энергия GPU);
   - предсказания на валидации.

2. **Informer + Optuna**  
   Внешний запуск оригинальной реализации Informer через обёртку и поиск гиперпараметров с Optuna.  
   Сохраняются:
   - метрики качества;
   - ресурсные метрики;
   - предсказания на валидации.

3. **LLM-агент**  
   Агент на основе LLM:
   - генерирует PyTorch-пайплайны для ETT-формата (кандидаты);
   - обучает каждый кандидат на обучающей выборке;
   - оценивает на валидации по MSE;
   - использует кроссовер лучших кандидатов для генерации новых.

В конце ноутбука строятся сравнительные таблицы и графики:

- сравнение MSE между всеми подходами;
- сравнение ресурсных метрик (время, память, энергия);
- динамика поиска LLM-агента по кандидатам.

Все пути и гиперпараметры задаются через переменные в первой кодовой ячейке и `.env` в корне проекта (`/edlm_search/.env`).


In [4]:
from __future__ import annotations

import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict
from typing import List
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv


def find_repo_root() -> Path:
    """Locate project root directory that contains 'src/edlm_search'."""
    current = Path.cwd().resolve()
    for candidate in (current, current.parent):
        if (candidate / 'src' / 'edlm_search').is_dir():
            return candidate
    raise RuntimeError('Cannot locate project root with "src/edlm_search" directory.')


REPO_ROOT: Path = find_repo_root()
SRC_DIR: Path = REPO_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ENV_PATH: Path = REPO_ROOT / '.env'
if ENV_PATH.is_file():
    load_dotenv(dotenv_path=ENV_PATH)

from edlm_search.experiments import (
    ExperimentResult,
    run_informer_optuna_etth_experiment,
    run_lstm_optuna_etth_experiment,
)
from edlm_search.experiments.datasets import load_ett_csv_dataset


@dataclass
class DatasetConfig:
    """Dataset configuration for ETTh experiments."""

    name: str
    csv_path: Path
    max_rows: int
    train_ratio: float


@dataclass
class LSTMOptunaConfig:
    """Configuration for LSTM + Optuna baseline."""
    n_trials: int
    target_column: str
    model_name: str
    artifacts_dir: Path
    device_type: str


@dataclass
class InformerOptunaConfig:
    """Configuration for Informer + Optuna baseline."""

    timeout_seconds: int
    n_trials: int
    metrics_root_dir: Path


@dataclass
class LLMSearchConfig:
    """Configuration for the LLM-based architecture search."""

    metric_name: str
    num_epochs_per_candidate: int
    num_initial_candidates: int
    num_crossover_candidates: int


@dataclass
class LLMProviderConfig:
    """Configuration of the LLM provider used by the agent."""

    provider: str
    base_url: str
    model_name: str
    temperature: float
    top_p: float
    api_key: str | None


def build_llm_provider_config_from_env() -> LLMProviderConfig:
    """Build LLMProviderConfig from environment variables."""
    provider_env = os.getenv('CHAT_CLIENT_PROVIDER', os.getenv('LLM_PROVIDER', 'lmstudio'))
    provider = provider_env.strip().lower()

    if provider == 'lmstudio':
        base_url = os.getenv('LM_STUDIO_BASE_URL', 'http://localhost:1234/v1')
        model_name = os.getenv('LM_STUDIO_MODEL_NAME', 'your-lmstudio-model-name')
        temperature_str = os.getenv('LM_STUDIO_TEMPERATURE', '0.2')
        top_p_str = os.getenv('LM_STUDIO_TOP_P', '0.95')
        api_key = None
    elif provider == 'deepseek':
        base_url = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com/v1')
        model_name = os.getenv('DEEPSEEK_MODEL_NAME', 'deepseek-coder')
        temperature_str = os.getenv('DEEPSEEK_TEMPERATURE', '0.4')
        top_p_str = os.getenv('DEEPSEEK_TOP_P', '0.9')
        api_key = os.getenv('DEEPSEEK_API_KEY')
    elif provider == 'openai':
        base_url = os.getenv('OPENAI_BASE_URL', 'https://api.openai.com/v1')
        model_name = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
        temperature_str = os.getenv('OPENAI_TEMPERATURE', '0.6')
        top_p_str = os.getenv('OPENAI_TOP_P', '0.85')
        api_key = os.getenv('OPENAI_API_KEY')
    else:
        raise ValueError(
                f'Unsupported LLM provider "{provider_env}". '
                f'Expected one of ["lmstudio", "deepseek", "openai"].'
        )

    if base_url is None or not base_url.strip():
        raise ValueError(f'Base URL must be provided for provider "{provider_env}".')
    if model_name is None or not model_name.strip():
        raise ValueError(f'Model name must be provided for provider "{provider_env}".')

    temperature = float(temperature_str)
    top_p = float(top_p_str)

    config = LLMProviderConfig(
            provider=provider,
            base_url=base_url.strip(),
            model_name=model_name.strip(),
            temperature=temperature,
            top_p=top_p,
            api_key=api_key,
    )
    return config


ETT_DATA_DIR: Path = SRC_DIR / 'ETDataset' / 'ETT-small'
ETTH1_PATH: Path = ETT_DATA_DIR / 'ETTh1.csv'
ETTH2_PATH: Path = ETT_DATA_DIR / 'ETTh2.csv'

MAX_ROWS: int = int(os.getenv('MAX_ROWS', '10000'))
TRAIN_RATIO: float = float(os.getenv('TRAIN_RATIO', '0.8'))

TARGET_COLUMN: str = os.getenv('TARGET_COLUMN', 'OT')

LSTM_N_TRIALS: int = int(os.getenv('N_TRIALS', '20'))
LSTM_MODEL_NAME: str = os.getenv('LSTM_MODEL_NAME', 'lstm-optuna')
ARTIFACTS_DIR: Path = (REPO_ROOT / os.getenv('ARTIFACTS_DIR', 'artifacts')).resolve()
DEVICE_TYPE: str = os.getenv('DEVICE_TYPE', 'auto')

INFORMER_TIMEOUT_SECONDS: int = int(os.getenv('INFORMER_TIMEOUT_SECONDS', '3600'))
INFORMER_N_TRIALS: int = int(os.getenv('INFORMER_N_TRIALS', '10'))
INFORMER_METRICS_ROOT: Path = (
        REPO_ROOT / os.getenv('INFORMER_METRICS_ROOT', 'artifacts/informer_optuna')
).resolve()

LLM_METRIC_NAME: str = os.getenv('AGENT_METRIC_NAME', os.getenv('LLM_METRIC_NAME', 'mse'))
LLM_NUM_EPOCHS_PER_CANDIDATE: int = int(
        os.getenv('AGENT_NUM_EPOCHS', os.getenv('LLM_NUM_EPOCHS', '5'))
)
LLM_NUM_INITIAL_CANDIDATES: int = int(
        os.getenv('AGENT_NUM_INITIAL_CANDIDATES', os.getenv('LLM_NUM_INITIAL_CANDIDATES', '2'))
)
LLM_NUM_CROSSOVER_CANDIDATES: int = int(
        os.getenv(
                'AGENT_NUM_CROSSOVER_CANDIDATES',
                os.getenv('LLM_NUM_CROSSOVER_CANDIDATES', '3'),
        )
)

STATEMENT_PATH: Path = SRC_DIR / 'statement.md'

dataset_configs: List[DatasetConfig] = [
    # DatasetConfig(
    #         name='ETTh1',
    #         csv_path=ETTH1_PATH,
    #         max_rows=MAX_ROWS,
    #         train_ratio=TRAIN_RATIO,
    # ),
    DatasetConfig(
            name='ETTh2',
            csv_path=ETTH2_PATH,
            max_rows=MAX_ROWS,
            train_ratio=TRAIN_RATIO,
    ),
]

lstm_optuna_config = LSTMOptunaConfig(
        n_trials=LSTM_N_TRIALS,
        target_column=TARGET_COLUMN,
        model_name=LSTM_MODEL_NAME,
        artifacts_dir=ARTIFACTS_DIR,
        device_type=DEVICE_TYPE,
)

informer_optuna_config = InformerOptunaConfig(
        timeout_seconds=INFORMER_TIMEOUT_SECONDS,
        n_trials=INFORMER_N_TRIALS,
        metrics_root_dir=INFORMER_METRICS_ROOT,
)

llm_search_config = LLMSearchConfig(
        metric_name=LLM_METRIC_NAME,
        num_epochs_per_candidate=LLM_NUM_EPOCHS_PER_CANDIDATE,
        num_initial_candidates=LLM_NUM_INITIAL_CANDIDATES,
        num_crossover_candidates=LLM_NUM_CROSSOVER_CANDIDATES,
)

llm_provider_config = build_llm_provider_config_from_env()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
INFORMER_METRICS_ROOT.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)
logger = logging.getLogger('etth_experiments')

logger.info(f'Repository root: {REPO_ROOT}')
logger.info(f'ETT data directory: {ETT_DATA_DIR}')
logger.info(f'Artifacts directory: {ARTIFACTS_DIR}')
logger.info(f'Informer metrics root: {INFORMER_METRICS_ROOT}')
logger.info(
        f'LLM provider configuration: provider={llm_provider_config.provider}, '
        f'base_url={llm_provider_config.base_url}, '
        f'model={llm_provider_config.model_name}, '
        f'temperature={llm_provider_config.temperature}, '
        f'top_p={llm_provider_config.top_p}'
)

2025-11-21 02:10:18,096 - INFO - etth_experiments - Repository root: /home/bulatov/LLM-NAS
2025-11-21 02:10:18,097 - INFO - etth_experiments - ETT data directory: /home/bulatov/LLM-NAS/src/ETDataset/ETT-small
2025-11-21 02:10:18,097 - INFO - etth_experiments - Artifacts directory: /home/bulatov/LLM-NAS/artifacts
2025-11-21 02:10:18,097 - INFO - etth_experiments - Informer metrics root: /home/bulatov/LLM-NAS/artifacts/informer_optuna
2025-11-21 02:10:18,098 - INFO - etth_experiments - LLM provider configuration: provider=openai, base_url=https://generativelanguage.googleapis.com/v1beta/openai/, model=gemini-2.5-flash, temperature=0.1, top_p=0.95


In [6]:
REPO_ROOT

PosixPath('/home/bulatov/LLM-NAS')

## 1. Загрузка и разбиение датасетов ETTh1 и ETTh2

На этом этапе выполняется только **предобработка данных**:

1. Чтение CSV-файлов `ETTh1.csv` и `ETTh2.csv`.
2. Ограничение числа строк `MAX_ROWS` (если задано).
3. Временное разбиение на обучающую и валидационную части по доле `TRAIN_RATIO`.

Модели здесь ещё не обучаются.


In [2]:
def load_single_dataset(config: DatasetConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load ETTh dataset from CSV and split into train and validation parts."""
    if not config.csv_path.is_file():
        raise FileNotFoundError(f'Dataset {config.name} not found at {config.csv_path}')
    train_df, valid_df = load_ett_csv_dataset(
            csv_path=str(config.csv_path),
            max_rows=config.max_rows,
            train_ratio=config.train_ratio,
    )
    logger.info(
            f'Dataset {config.name} loaded: train_rows={len(train_df)}, '
            f'valid_rows={len(valid_df)}, max_rows={config.max_rows}, '
            f'train_ratio={config.train_ratio}'
    )
    return train_df, valid_df


def load_all_datasets(configs: List[DatasetConfig]) -> Tuple[Dict[str, pd.DataFrame], Dict[str, pd.DataFrame]]:
    """Load and split all configured datasets."""
    train_dfs: Dict[str, pd.DataFrame] = {}
    valid_dfs: Dict[str, pd.DataFrame] = {}
    for cfg in configs:
        train_df, valid_df = load_single_dataset(cfg)
        train_dfs[cfg.name] = train_df
        valid_dfs[cfg.name] = valid_df
    return train_dfs, valid_dfs


train_dfs, valid_dfs = load_all_datasets(dataset_configs)
train_dfs.keys(), valid_dfs.keys()


2025-11-21 01:56:21,257 - INFO - etth_experiments - Dataset ETTh2 loaded: train_rows=4000, valid_rows=1000, max_rows=5000, train_ratio=0.8


(dict_keys(['ETTh2']), dict_keys(['ETTh2']))

## 2. LSTM + Optuna: поиск гиперпараметров и обучение лучшей модели

На этом этапе:

1. Для каждого датасета (ETTh1, ETTh2) запускается **Optuna-поиск гиперпараметров** LSTM:
   - длина скрытого состояния;
   - число слоёв;
   - скорость обучения;
   - размер батча.

2. Для лучшей конфигурации выполняется **обучение модели LSTM** с заданным числом эпох `NUM_EPOCHS`.

3. После обучения:
   - считается MSE на валидации;
   - собираются ресурсные метрики (wall-time, CPU, память, энергия GPU);
   - сохраняются предсказания на валидации в `ARTIFACTS_DIR`.

Эта ячейка отвечает за **обучение и валидацию LSTM-модели**.


In [ ]:
def run_lstm_optuna_for_all_datasets(
        configs: List[DatasetConfig],
        lstm_config: LSTMOptunaConfig,
) -> Dict[str, ExperimentResult]:
    """Run LSTM + Optuna baseline for all datasets and return results."""
    results: Dict[str, ExperimentResult] = {}
    for cfg in configs:
        if not cfg.csv_path.is_file():
            logger.info(
                    f'[LSTM+Optuna] Dataset {cfg.name} skipped: missing CSV at {cfg.csv_path}'
            )
            continue
        logger.info(
                f'[LSTM+Optuna] Starting search for dataset={cfg.name}, '
                f'n_trials={lstm_config.n_trials}, '
                f'device_type={lstm_config.device_type}'
        )
        result = run_lstm_optuna_etth_experiment(
                dataset_name=cfg.name,
                csv_path=str(cfg.csv_path),
                max_rows=cfg.max_rows,
                train_ratio=cfg.train_ratio,
                n_trials=lstm_config.n_trials,
                target_column=lstm_config.target_column,
                model_name=lstm_config.model_name,
                artifacts_dir=str(lstm_config.artifacts_dir),
                device_type=lstm_config.device_type,
        )
        results[cfg.name] = result
        mse_value = float(result.metrics.get('mse', float('nan')))
        logger.info(
                f'[LSTM+Optuna] Finished for dataset={cfg.name}, mse={mse_value}'
        )
    return results


lstm_optuna_results = run_lstm_optuna_for_all_datasets(
        configs=dataset_configs,
        lstm_config=lstm_optuna_config,
)
lstm_optuna_results


### 2.1. Визуализация предсказаний LSTM на валидации

На этом этапе **обучение не выполняется**.  

Мы:

1. Читаем сохранённые CSV с предсказаниями LSTM для валидации.  
2. Строим графики `y_true` vs `y_pred` для первых `N` точек.

Это помогает визуально оценить качество прогноза.


In [ ]:
def load_predictions_csv(path: Path) -> pd.DataFrame:
    """Load validation predictions CSV with columns 'y_true' and 'y_pred'."""
    if not path.is_file():
        raise FileNotFoundError(f'Predictions CSV not found at {path}')
    df = pd.read_csv(path)
    if 'y_true' not in df.columns or 'y_pred' not in df.columns:
        raise ValueError(
                f'Predictions CSV at {path} must contain columns "y_true" and "y_pred".'
        )
    return df


def plot_validation_predictions(
        df: pd.DataFrame,
        title: str,
        max_points: int,
) -> None:
    """Plot ground truth and predictions for validation data."""
    subset = df.head(max_points)
    index = np.arange(len(subset))
    plt.figure(figsize=(8, 4))
    plt.plot(index, subset['y_true'], label='y_true')
    plt.plot(index, subset['y_pred'], label='y_pred')
    plt.xlabel('time step (validation index)')
    plt.ylabel(TARGET_COLUMN)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


for cfg in dataset_configs:
    dataset_name = cfg.name
    predictions_path = ARTIFACTS_DIR / f'{lstm_optuna_config.model_name}_{dataset_name}_valid_predictions.csv'
    if not predictions_path.is_file():
        logger.info(
                f'[{dataset_name}] LSTM predictions CSV not found at {predictions_path}; skipping plot.'
        )
        continue
    df_pred = load_predictions_csv(predictions_path)
    plot_validation_predictions(
            df=df_pred,
            title=f'LSTM+Optuna validation predictions on {dataset_name}',
            max_points=200,
    )


## 3. Informer + Optuna: внешний запуск и подбор гиперпараметров

На этом этапе:

1. Для каждого датасета запускается **Informer + Optuna** через обёртку `Informer2020/informer_experiment_wrapper.py`.
2. Optuna подбирает гиперпараметры Informer (размеры слоёв, число голов, dropout и т.д.).
3. Для лучшей конфигурации внешняя реализация Informer:
   - обучается заданное число эпох `INFORMER_NUM_EPOCHS`;
   - вычисляет метрики на валидации;
   - сохраняет предсказания и служебные файлы.

Эксперимент оборачивается мониторингом ресурсов, чтобы собрать единую статистику по времени и потреблению ресурсов.


In [ ]:
def run_informer_optuna_for_all_datasets(
        configs: List[DatasetConfig],
        informer_config: InformerOptunaConfig,
) -> Dict[str, ExperimentResult]:
    """Run Informer + Optuna baseline for all datasets and return results."""
    results: Dict[str, ExperimentResult] = {}
    for cfg in configs:
        if not cfg.csv_path.is_file():
            logger.info(
                    f'[Informer+Optuna] Dataset {cfg.name} skipped: missing CSV at {cfg.csv_path}'
            )
            continue

        metrics_root_for_dataset = informer_config.metrics_root_dir / cfg.name
        metrics_root_for_dataset.mkdir(parents=True, exist_ok=True)

        logger.info(
                f'[Informer+Optuna] Starting search for dataset={cfg.name}, '
                f'n_trials={informer_config.n_trials}, '
                f'timeout_seconds={informer_config.timeout_seconds}'
        )

        result = run_informer_optuna_etth_experiment(
                dataset_name=cfg.name,
                csv_path=str(cfg.csv_path),
                informer_script_path=str(
                        SRC_DIR / 'Informer2020' / 'informer_experiment_wrapper.py'
                ),
                metrics_root_dir=str(metrics_root_for_dataset),
                base_extra_args=None,
                timeout_seconds=informer_config.timeout_seconds,
                model_name=f'informer-optuna-{cfg.name.lower()}',
                n_trials=informer_config.n_trials,
        )
        results[cfg.name] = result
        mse_value = float(result.metrics.get('mse', float('nan')))
        logger.info(
                f'[Informer+Optuna] Finished for dataset={cfg.name}, mse={mse_value}'
        )
    return results


informer_optuna_results = run_informer_optuna_for_all_datasets(
        configs=dataset_configs,
        informer_config=informer_optuna_config,
)
informer_optuna_results

### 3.1. Визуализация предсказаний Informer + Optuna на валидации

Здесь:

1. Берётся лучший запуск Informer после Optuna-поиска.
2. Читаются сохранённые предсказания на валидации (путь берётся из `extra_info['predictions_csv_path']`).
3. Строятся графики `y_true` vs `y_pred`.

На этом этапе обучения нет, только анализ предсказаний.


In [ ]:
for cfg in dataset_configs:
    dataset_name = cfg.name
    informer_result = informer_optuna_results.get(dataset_name)
    if informer_result is None:
        logger.info(
                f'[{dataset_name}] Informer+Optuna result is missing; skipping predictions plot.'
        )
        continue
    predictions_csv_path_value = informer_result.extra_info.get('predictions_csv_path')
    if not isinstance(predictions_csv_path_value, str):
        logger.info(
                f'[{dataset_name}] Informer+Optuna predictions path is missing; skipping predictions plot.'
        )
        continue
    predictions_path = Path(predictions_csv_path_value)
    if not predictions_path.is_file():
        logger.info(
                f'[{dataset_name}] Informer+Optuna predictions CSV not found at {predictions_path}; skipping plot.'
        )
        continue
    df_pred = load_predictions_csv(predictions_path)
    plot_validation_predictions(
            df=df_pred,
            title=f'Informer+Optuna validation predictions on {dataset_name}',
            max_points=200,
    )


## 4. LLM-агент: генерация и оценка архитектур

LLM-агент строится поверх существующей инфраструктуры:

- `statement.md` (в `/edlm_search/src/statement.md`) описывает постановку задачи;  
  текст используется для формирования промптов.
- `LLMPipeline` отвечает за взаимодействие с LLM и парсинг XML-ответов в файлы (`main.py`, `model_config.json`, `training_args.json`).
- `Candidate` инкапсулирует сгенерированную идею и файлы.
- `UnsafeRunner` запускает код кандидата в отдельном процессе.
- `ETTEvaluator` реализует цикл обучения и оценки кандидата, возвращая MSE и вспомогательные метрики.

Логика поиска:

1. Сначала генерируется несколько **начальных кандидатов** из шаблона `new_candidate`.
2. Каждый кандидат **обучается и оценивается** на фиксированном числе эпох `LLM_NUM_EPOCHS`.
3. Лучшие кандидаты используются как родители в шаблоне `crossover_candidate`:
   - LLM получает идеи, метрики и файлы родителей;
   - генерирует новый, объединённый пайплайн.
4. Дети также обучаются и оцениваются.
5. Для каждого датасета сохраняем таблицу кандидатов и выбираем лучшего по MSE.

Далее — реализация этой логики.


In [ ]:
from pathlib import Path

from edlm_search.agent.llm_search_agent import (
    AgentCandidateRecord,
    build_problem,
    create_llm_pipeline,
    run_llm_search_for_all_datasets,
    LLMSearchRun,
)

problem_instance = build_problem(STATEMENT_PATH)
llm_pipeline = create_llm_pipeline(llm_provider_config)

llm_search_run: LLMSearchRun = await run_llm_search_for_all_datasets(
        dataset_configs=dataset_configs,
        train_dfs=train_dfs,
        valid_dfs=valid_dfs,
        problem=problem_instance,
        llm_pipeline=llm_pipeline,
        config=llm_search_config,
        target_column=TARGET_COLUMN,
        provider_config=llm_provider_config,
)

# The results are now attributes of llm_search_run
llm_search_results = llm_search_run.results
llm_best_mse = llm_search_run.best_metrics

2025-11-21 01:56:33,727 - INFO - edlm_search.agent.llm_search_agent - Problem statement loaded from /home/bulatov/LLM-NAS/src/statement.md
2025-11-21 01:56:33,857 - INFO - edlm_search.agent.llm_search_agent - LLM pipeline created for provider="openai", model="gemini-2.5-flash", base_url="https://generativelanguage.googleapis.com/v1beta/openai/", temperature=0.1, top_p=0.95
2025-11-21 01:56:33,888 - INFO - edlm_search.agent.llm_search_agent - [LLM agent] Global search started for 1 datasets with torch_backend="cuda".
2025-11-21 01:56:33,889 - INFO - edlm_search.agent.llm_search_agent - [LLM agent] Starting search on dataset="ETTh2".
2025-11-21 01:56:33,889 - INFO - edlm_search.agent.llm_search_agent - [LLM agent] Dataset="ETTh2", initial_candidates=5, crossover_candidates=2, epochs_per_candidate=5, torch_backend="cuda"
2025-11-21 01:56:33,889 - INFO - edlm_search.agent.llm_search_agent - [LLM agent] Generating initial candidate_id=0 for dataset="ETTh2".
[proxychains] Strict chain  ...  

<idea>
The solution employs a Transformer Encoder model for multivariate long-sequence time-series forecasting, using a custom Min-Max scaler for data normalization and a recursive single-step prediction strategy during validation to generate a full sequence of predictions.
</idea>
<files>
    <file path="main.py">
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json
import math

# --- Device Configuration ---
device = torch.device("cuda")

# --- Custom MinMaxScaler (since sklearn is not allowed) ---
class CustomMinMaxScaler:
    def __init__(self):
        self.min_vals = None
        self.max_vals = None
        self.feature_names = None

    def fit(self, data):
        if not isinstance(data, pd.DataFrame):
            raise ValueError("Input data for scaler must be a Pandas DataFrame.")
        
        self.feature_names = data.columns.tolist()
        self.min_vals = dat

2025-11-21 01:58:09,191 - INFO - edlm_search.agent.ett_evaluator - Получено время старта процесса кандидата: 2025-11-21T01:58:09.191104
2025-11-21 01:58:09,500 - ERROR - edlm_search.agent.llm_search_agent - [LLM agent] Candidate evaluation failed on dataset="ETTh2", candidate_index=0, error=KeyError: "['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL'] not in index"
Traceback (most recent call last):
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/llm_search_agent.py", line 382, in _run_single_candidate_flow
    record = await evaluate_single_candidate(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
    )
    ^
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/llm_search_agent.py", line 178, in evaluate_single_candidate
    metrics = await evaluator.evaluate(runner=runner, candidate=candidate)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/ett_evaluator.py", line 70, in evaluate
    as

<idea>The solution employs a Transformer Encoder model for multivariate long-sequence time-series forecasting, using a custom Min-Max scaler for data normalization and a recursive single-step prediction strategy during validation to generate a full sequence of predictions, with robust handling for dataset column consistency and single-feature inverse scaling.</idea>
<files>
    <file path="main.py">
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json
import math

# --- Device Configuration ---
device = torch.device("cuda")

# --- Custom MinMaxScaler (since sklearn is not allowed) ---
class CustomMinMaxScaler:
    def __init__(self):
        self.min_vals = None
        self.max_vals = None
        self.feature_names = None

    def fit(self, data):
        if not isinstance(data, pd.DataFrame):
            raise ValueError("Input data for scaler must be a Pandas DataFrame.")
 

2025-11-21 01:59:17,491 - INFO - edlm_search.agent.ett_evaluator - Получено время старта процесса кандидата: 2025-11-21T01:59:17.491490
2025-11-21 01:59:17,812 - ERROR - edlm_search.agent.llm_search_agent - [LLM agent] Candidate evaluation failed on dataset="ETTh2", candidate_index=0, error=TypeError: CustomMinMaxScaler.transform() got an unexpected keyword argument 'feature_name'
Traceback (most recent call last):
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/llm_search_agent.py", line 382, in _run_single_candidate_flow
    record = await evaluate_single_candidate(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
    )
    ^
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/llm_search_agent.py", line 178, in evaluate_single_candidate
    metrics = await evaluator.evaluate(runner=runner, candidate=candidate)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/ett_evaluator.py", line 70, 

...

2025-11-21 02:02:28,634 - INFO - edlm_search.agent.ett_evaluator - Получено время старта процесса кандидата: 2025-11-21T02:02:28.634528
2025-11-21 02:02:30,170 - ERROR - edlm_search.agent.llm_search_agent - [LLM agent] Candidate evaluation failed on dataset="ETTh2", candidate_index=1, error=ValueError: could not broadcast input array from shape (24,) into shape (904,)
Traceback (most recent call last):
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/llm_search_agent.py", line 382, in _run_single_candidate_flow
    record = await evaluate_single_candidate(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
    )
    ^
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/llm_search_agent.py", line 178, in evaluate_single_candidate
    metrics = await evaluator.evaluate(runner=runner, candidate=candidate)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bulatov/LLM-NAS/src/edlm_search/agent/ett_evaluator.py", line 70, in evaluate
 

<idea>
The solution employs an LSTM-based sequence-to-sequence model for long-sequence time-series forecasting, leveraging a custom data scaler and time-based feature engineering, and robustly handles validation predictions by returning historical ground truth for context and model forecasts for the future horizon, adapting to varying forecast horizon lengths by filling beyond the model's prediction capacity with the last predicted value.
</idea>
<files>
    <file path="main.py">
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import json

# --- 1. Custom StandardScaler (sklearn is not allowed) ---
class CustomStandardScaler:
    """
    A custom StandardScaler to normalize data.
    """
    def __init__(self):
        self.mean = None
        self.std = None
        self.is_fitted = False

    def fit(self, data):
        """
        Fits the scaler to the provided data.
        data: A numpy array or pandas Da

2025-11-21 02:03:45,796 - INFO - edlm_search.agent.ett_evaluator - Получено время старта процесса кандидата: 2025-11-21T02:03:45.796731
2025-11-21 02:03:47,193 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:03:47,603 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:03:48,012 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:03:48,422 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:03:48,834 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:03:48,836 - INFO - edlm_search.agent.ett_evaluator - Получена оценка энергопотребления 207.0440 Дж.
2025-11-21 02:03:49,420 - INFO - edlm_search.agent.ett_evaluator - Оценка кандидата "The solution employs an LSTM-based sequence-to-sequence mode

<idea>
A sequence-to-sequence LSTM model with an auto-regressive prediction strategy, trained to forecast the next hour's oil temperature using a look-back window of all available features, including the oil temperature itself.
</idea>
<files>
    <file path="main.py">
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import json
import os

# --- Device Configuration ---
DEVICE = torch.device("cuda")

# --- Custom MinMaxScaler (since sklearn is not allowed) ---
class CustomMinMaxScaler:
    """
    A custom implementation of MinMaxScaler for numerical stability and to avoid
    relying on external libraries like scikit-learn.
    """
    def __init__(self):
        self.min_vals = None
        self.max_vals = None

    def fit(self, data):
        """
        Fits the scaler to the provided data by calculating min and max values
        for each feature.
        Args:
            data (np.ndarray): The input data 

2025-11-21 02:05:31,577 - INFO - edlm_search.agent.ett_evaluator - Получено время старта процесса кандидата: 2025-11-21T02:05:31.576924
2025-11-21 02:05:33,552 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:05:34,526 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:05:35,504 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:05:36,477 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:05:37,446 - INFO - edlm_search.agent.ett_evaluator - Получены валидационные предсказания из 1000 значений.
2025-11-21 02:05:37,447 - INFO - edlm_search.agent.ett_evaluator - Получена оценка энергопотребления 374.1660 Дж.
2025-11-21 02:05:38,037 - INFO - edlm_search.agent.ett_evaluator - Оценка кандидата "A sequence-to-sequence LSTM model with an auto-regressive pr

### 4.1. Таблица всех кандидатов LLM-агента

Теперь переведём результаты работы агента в `pandas.DataFrame`:

- каждая строка — отдельный кандидат;
- столбцы: датасет, `candidate_id`, идея, метрики (MSE, энергия, время и т.п.).

Таблица также сохраняется в `ARTIFACTS_DIR` для последующего анализа.


In [4]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path
import pandas as pd
from typing import Dict, List, TYPE_CHECKING, Protocol

# This block is for type hinting and may not be necessary in all notebook environments
# If you get errors, you might need to install the libraries or adjust the imports.
try:
    from src.edlm_search.agent.llm_search_agent import LLMSearchRun
except ImportError:
    # Define protocol as a fallback for type hinting if the main class is not available
    class LLMProviderConfigProtocol(Protocol):
        provider: str
        base_url: str
        model_name: str
        temperature: float
        top_p: float

    class LLMSearchConfigProtocol(Protocol):
        num_initial_candidates: int
        num_crossover_candidates: int
        num_epochs_per_candidate: int
        metric_name: str

    class CandidateProtocol(Protocol):
        files: Dict[str, str]

    class AgentCandidateRecordProtocol(Protocol):
        candidate_id: int
        idea: str
        metrics: Dict[str, float] | None
        candidate: CandidateProtocol
        metadata: dict[str, object]

    class LLMSearchRun(Protocol):
        results: Dict[str, List[AgentCandidateRecordProtocol]]
        best_metrics: Dict[str, float]
        provider_config: LLMProviderConfigProtocol
        search_config: LLMSearchConfigProtocol


def build_llm_results_dataframe(
        search_run: LLMSearchRun,
) -> pd.DataFrame:
    """Convert LLM search results from an LLMSearchRun to a flat pandas.DataFrame."""
    rows: List[Dict[str, object]] = []
    model_name = search_run.provider_config.model_name
    records_per_dataset = search_run.results

    for dataset_name, records in records_per_dataset.items():
        for record in records:
            # Base information
            row: Dict[str, object] = {
                'dataset': dataset_name,
                'candidate_id': record.candidate_id,
                'idea': record.idea,
                'model_name': model_name,
            }

            # Add metadata from the record
            row['fix_attempts'] = record.metadata.get('fix_attempts', 0)
            row['last_error'] = record.metadata.get('last_error', None)
            row['status'] = record.metadata.get('status', 'success' if record.metrics else 'failed')

            # Add placeholder token counts
            row['total_input_tokens'] = 0
            row['total_output_tokens'] = 0

            # Add candidate files as JSON string
            row['candidate_files_json'] = json.dumps(record.candidate.files)

            # Safely add metrics
            if record.metrics is not None:
                for metric_name, metric_value in record.metrics.items():
                    row[metric_name] = float(metric_value)

            rows.append(row)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    # Define a consistent column order
    core_cols = [
        'dataset', 'candidate_id', 'idea', 'status', 'model_name',
        'fix_attempts', 'last_error'
    ]
    metric_cols = [col for col in df.columns if col not in core_cols and col not in ['candidate_files_json', 'total_input_tokens', 'total_output_tokens']]
    token_cols = ['total_input_tokens', 'total_output_tokens']
    files_col = ['candidate_files_json']

    all_cols = core_cols + metric_cols + token_cols + files_col
    for col in all_cols:
        if col not in df.columns:
            df[col] = None

    df = df[all_cols]
    df.sort_values(by=['dataset', 'candidate_id'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


def save_run_artifacts(search_run: LLMSearchRun, artifacts_base_dir: str | Path) -> None:
    """
    Saves all artifacts from an LLMSearchRun to a structured directory.
    For each dataset, a unique directory is created containing the run's config,
    the best candidate's files, and a CSV of all candidates.
    """
    base_dir = Path(artifacts_base_dir)
    base_dir.mkdir(exist_ok=True)

    run_config = {
        'provider_config': {
            'provider': search_run.provider_config.provider,
            'base_url': search_run.provider_config.base_url,
            'model_name': search_run.provider_config.model_name,
            'temperature': search_run.provider_config.temperature,
            'top_p': search_run.provider_config.top_p,
        },
        'search_config': {
            'num_initial_candidates': search_run.search_config.num_initial_candidates,
            'num_crossover_candidates': search_run.search_config.num_crossover_candidates,
            'num_epochs_per_candidate': search_run.search_config.num_epochs_per_candidate,
            'metric_name': search_run.search_config.metric_name,
        },
        'best_metrics_overall': search_run.best_metrics,
    }
    model_name_sanitized = run_config['provider_config']['model_name'].replace('/', '_')

    full_df = build_llm_results_dataframe(search_run)

    for dataset_name, records in search_run.results.items():
        timestamp = datetime.now().strftime('%Y-%m-%d_%H:%M:%S')
        run_dir_name = f"agent_run_{dataset_name}_{MAX_ROWS}_{model_name_sanitized}_{timestamp}"
        run_dir = base_dir / run_dir_name
        run_dir.mkdir(exist_ok=False)

        config_path = run_dir / 'run_config.json'
        with config_path.open('w', encoding='utf-8') as f:
            json.dump(run_config, f, indent=4)

        if not full_df.empty:
            dataset_df = full_df[full_df['dataset'] == dataset_name]
            if not dataset_df.empty:
                csv_path = run_dir / 'candidate_records.csv'
                dataset_df.to_csv(csv_path, index=False)

        successful_records = [r for r in records if r.metrics is not None]
        if successful_records:
            metric_name = search_run.search_config.metric_name
            best_record = min(
                successful_records,
                key=lambda r: r.metrics.get(metric_name, float('inf')),
            )
            best_candidate = best_record.candidate
            for filename, content in best_candidate.files.items():
                file_path = run_dir / filename
                if filename == 'main.py':
                    idea = '# ' + best_candidate.idea.replace('\n', '\n# ') + '\n'
                    content = idea + content
                file_path.write_text(content, encoding='utf-8')

        print(f"Artifacts for dataset '{dataset_name}' saved to: {run_dir}")


ARTIFACTS_DIR = Path("../artifacts/")
save_run_artifacts(llm_search_run, ARTIFACTS_DIR)


Artifacts for dataset 'ETTh2' saved to: ../artifacts/agent_run_ETTh2_5000_gemini-2.5-flash_2025-11-21_01:54:45


## 5. Сравнение подходов по MSE и ресурсным метрикам

На этом этапе объединяем результаты:

- LSTM + Optuna;
- Informer + Optuna;
- LLM-агент (лучший кандидат).

Далее строятся:

1. Таблица с MSE по каждому подходу и датасету.
2. Столбчатые диаграммы MSE.
3. Таблица ресурсных метрик (время, память, энергия GPU).
4. Диаграммы сравнения ресурсных метрик.
5. График "MSE vs wall time".

Обучения на этом шаге не происходит — только анализ уже полученных результатов.


In [ ]:
def build_comparison_table(
        configs: List[DatasetConfig],
        lstm_results: Dict[str, ExperimentResult],
        informer_results: Dict[str, ExperimentResult],
        llm_best: Dict[str, float],
        primary_metric: str,
) -> pd.DataFrame:
    """Build comparison table across approaches for all datasets."""
    rows: List[Dict[str, object]] = []
    for cfg in configs:
        dataset_name = cfg.name

        lstm_result = lstm_results.get(dataset_name)
        if lstm_result is not None:
            mse_value = float(lstm_result.metrics.get(primary_metric, float('nan')))
            rows.append(
                    {
                        'dataset': dataset_name,
                        'approach': 'lstm_optuna_best',
                        primary_metric: mse_value,
                    }
            )

        informer_result = informer_results.get(dataset_name)
        if informer_result is not None:
            informer_mse_raw = informer_result.metrics.get(primary_metric)
            informer_mse = (
                float(informer_mse_raw)
                if informer_mse_raw is not None
                else float('nan')
            )
            rows.append(
                    {
                        'dataset': dataset_name,
                        'approach': 'informer_optuna_best',
                        primary_metric: informer_mse,
                    }
            )

        llm_mse = llm_best.get(dataset_name)
        if llm_mse is not None:
            rows.append(
                    {
                        'dataset': dataset_name,
                        'approach': 'llm_agent_best_candidate',
                        primary_metric: float(llm_mse),
                    }
            )

    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df.sort_values(by=['dataset', 'approach'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


comparison_df = build_comparison_table(
        configs=dataset_configs,
        lstm_results=lstm_optuna_results,
        informer_results=informer_optuna_results,
        llm_best=llm_best_mse,
        primary_metric=LLM_METRIC_NAME,
)
comparison_csv_path = ARTIFACTS_DIR / 'etth_comparison_metrics.csv'
if not comparison_df.empty:
    comparison_df.to_csv(comparison_csv_path, index=False)
    logger.info(
            f'Comparison metrics table saved to {comparison_csv_path}'
    )
comparison_df


In [ ]:
if comparison_df.empty:
    logger.info('Comparison dataframe is empty; skipping MSE bar plots.')
else:
    metric_name = LLM_METRIC_NAME
    for cfg in dataset_configs:
        dataset_name = cfg.name
        subset = comparison_df[comparison_df['dataset'] == dataset_name]
        if subset.empty:
            logger.info(
                    f'[{dataset_name}] Comparison subset is empty; skipping MSE bar plot.'
            )
            continue

        plt.figure(figsize=(6, 4))
        x_positions = np.arange(len(subset))
        plt.bar(x_positions, subset[metric_name])
        plt.xticks(
                x_positions,
                subset['approach'],
                rotation=30,
                ha='right',
        )
        plt.ylabel(metric_name)
        plt.title(f'MSE comparison on {dataset_name}')
        plt.tight_layout()
        plt.show()


In [ ]:
def build_resource_table(
        configs: List[DatasetConfig],
        lstm_results: Dict[str, ExperimentResult],
        informer_results: Dict[str, ExperimentResult],
        primary_metric: str,
) -> pd.DataFrame:
    """Build table with resource metrics for LSTM and Informer baselines."""
    rows: List[Dict[str, object]] = []

    def append_row(dataset_name: str, approach: str, result: ExperimentResult) -> None:
        metrics = result.metrics
        row: Dict[str, object] = {
            'dataset': dataset_name,
            'approach': approach,
            primary_metric: float(metrics.get(primary_metric, float('nan'))),
            'wall_seconds_total': float(metrics.get('wall_seconds_total', float('nan'))),
            'cpu_user_seconds_total': float(
                    metrics.get('cpu_user_seconds_total', float('nan'))
            ),
            'cpu_system_seconds_total': float(
                    metrics.get('cpu_system_seconds_total', float('nan'))
            ),
            'rss_mb_delta': float(metrics.get('rss_mb_delta', float('nan'))),
            'gpu_total_energy_joules': float(
                    metrics.get('gpu_total_energy_joules', float('nan'))
            ),
        }
        rows.append(row)

    for cfg in configs:
        dataset_name = cfg.name
        lstm_result = lstm_results.get(dataset_name)
        if lstm_result is not None:
            append_row(dataset_name, 'lstm_optuna_best', lstm_result)

        informer_result = informer_results.get(dataset_name)
        if informer_result is not None:
            append_row(dataset_name, 'informer_optuna_best', informer_result)

    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df.sort_values(by=['dataset', 'approach'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


resource_df = build_resource_table(
        configs=dataset_configs,
        lstm_results=lstm_optuna_results,
        informer_results=informer_optuna_results,
        primary_metric=LLM_METRIC_NAME,
)
resource_df


In [ ]:
if resource_df.empty:
    logger.info('Resource metrics dataframe is empty; skipping resource plots.')
else:
    metrics_to_plot = ['wall_seconds_total', 'gpu_total_energy_joules', 'rss_mb_delta']
    for cfg in dataset_configs:
        dataset_name = cfg.name
        subset = resource_df[resource_df['dataset'] == dataset_name]
        if subset.empty:
            logger.info(
                    f'[{dataset_name}] Resource subset is empty; skipping resource plots.'
            )
            continue

        for metric_name in metrics_to_plot:
            plt.figure(figsize=(6, 4))
            x_positions = np.arange(len(subset))
            plt.bar(x_positions, subset[metric_name])
            plt.xticks(
                    x_positions,
                    subset['approach'],
                    rotation=30,
                    ha='right',
            )
            plt.ylabel(metric_name)
            plt.title(f'{metric_name} comparison on {dataset_name}')
            plt.tight_layout()
            plt.show()


In [ ]:
if resource_df.empty:
    logger.info('Resource metrics dataframe is empty; skipping MSE vs wall time plot.')
else:
    plt.figure(figsize=(6, 4))
    for cfg in dataset_configs:
        dataset_name = cfg.name
        subset = resource_df[resource_df['dataset'] == dataset_name]
        if subset.empty:
            continue
        plt.scatter(
                subset['wall_seconds_total'],
                subset[LLM_METRIC_NAME],
                label=dataset_name,
        )
    plt.xlabel('wall_seconds_total')
    plt.ylabel(LLM_METRIC_NAME)
    plt.title('MSE vs wall time across datasets and approaches')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if llm_candidates_df.empty:
    logger.info('LLM candidates dataframe is empty; skipping LLM dynamics plots.')
else:
    for cfg in dataset_configs:
        dataset_name = cfg.name
        subset = llm_candidates_df[llm_candidates_df['dataset'] == dataset_name]
        if subset.empty:
            logger.info(
                    f'[{dataset_name}] No LLM candidates found; skipping dynamics plot.'
            )
            continue

        subset_sorted = subset.sort_values(by='candidate_id')
        plt.figure(figsize=(6, 4))
        plt.plot(
                subset_sorted['candidate_id'],
                subset_sorted[LLM_METRIC_NAME],
                marker='o',
        )
        plt.xlabel('candidate_id')
        plt.ylabel(LLM_METRIC_NAME)
        plt.title(
                f'LLM agent search dynamics on {dataset_name}: {LLM_METRIC_NAME} per candidate'
        )
        plt.grid(True)
        plt.tight_layout()
        plt.show()